# മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് — ആസ്യൂർ ഓപ്പൺഎഐ (Responses API)

ഈ കോഡ് സാമ്പിളിൽ, **Microsoft Agent Framework (MAF)** ഉപയോഗിച്ച് **Azure OpenAI** പിന്തുണയുള്ള ഒരു സിമ്പിൾ ഏജന്റ് **Responses API** ഉപയോഗിച്ച് നിർമിക്കുന്നുവെന്ന് കാണിക്കും.

> **മൈഗ്രേഷൻ കുറിപ്പ്:** ഈ സാമ്പിൾ മുൻപ് GitHub മോഡലുകളോടെ Semantic Kernel ഉപയോഗിച്ചിരുന്നു. ഇത് മൈഗ്രേറ്റുചെയ്‌ത ശേഷം Microsoft Agent Framework ആക്കി മാറ്റപ്പെട്ടിരിക്കുന്നു, കൂടാതെ GitHub മോഡലുകൾ (പഴക്കം ചെന്നത്, 2026 ജൂലൈയിൽ വിരമിക്കും) Azure OpenAI കൊണ്ട് മാറ്റിവച്ചു, ഇത് Responses API പിന്തുണക്കുന്നു. MAFയിലെ `OpenAIChatClient` ആസ്യൂർ ഓപ്പൺഎഐയുടെ സ്ഥിരതയുള്ള `/openai/v1/` എਂഡ്പോയിന്റ് ലക്ഷ്യമിട്ട് Responses API സ്വതവേയായി ഉപയോഗിക്കുന്നു.

ഈ സാമ്പിളിന്റെ ഉദ്ദേശ്യം, പിന്നീട് വിവിധ ഏജന്റിക് പാറ്റേണുകൾ നടപ്പിലാക്കുമ്പോൾ മറ്റ് കോഡ് സാമ്പിളുകളിൽ പ്രയോഗിക്കാവുന്ന ഘട്ടങ്ങൾ കാണിക്കുക ആണ്.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ആവശ്യമായ Python പാക്കേജുകൾ ഇറക്കുമതി ചെയ്യുക


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ഒരു ഉപകരണത്തെ നിർവചിക്കൽ

Microsoft Agent Framework-ൽ, **അപേക്ഷകയ്ക്ക്** വിളിക്കാനാവുന്ന `@tool` കൊണ്ട് അലങ്കരിച്ചൊരു സാധാരണ Python ഫങ്ഷൻ ആണ് ഒരു **ഉപകരണം**. താഴെ ഒരു ഉപകരണത്തെ നിർവചിക്കുന്നു, അത് ഒരു യാദൃച്ഛിക അവധിക്കാല ഡെസ്റ്റിനേഷൻ നൽകുന്നു, പണ്ടത്തെത് ആവർത്തിക്കുന്നത് ഒഴിവാക്കുന്നു.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Creating the Agent

Here, we create the Agent named `TravelAgent`.

In this example, we use very basic instructions. Feel free to modify these instructions to observe how the agent's behavior changes.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ഏജന്റ് റൺ ചെയ്യുന്നു

ഇനി നാം ഏജന്റ് റൺചെയ്യാം. ഏജന്റ് സംഭാഷണത്തിന് ഇടയിലുള്ള ടേൺസ് ഓർക്കാൻ `AgentSession` സൃഷ്ടിക്കുന്നു, പിന്നെ രണ്ട് `user_inputs` അയയ്ക്കുന്നു. ആദ്യത്തേതിൽ ഒരു യാത്ര ചോദിക്കുന്നു; രണ്ടാമത്തേതിൽ ഉപയോക്താവ് നിർദ്ദേശം മോശമാണെന്ന് പറഞ്ഞു മറ്റൊരു യാത്ര ചോദിക്കുന്നു — ഏജന്റ് സെഷൻ ചരിത്രവും `get_random_destination` ഉപകരണവും ഉപയോഗിച്ച് പ്രതികരിക്കുന്നു.

ഏജന്റ് വ്യത്യസ്തമായി പ്രതികരിക്കുന്നത് കാണാൻ ഈ സന്ദേശങ്ങൾ നിങ്ങൾ മാറ്റി നോക്കാം. പ്രതികരണങ്ങൾ **streamed** ടോക്കൺ-തോക്കണായി ലഭിക്കും.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
